# 09 · 3D scenes, terrain intuition, and time


**Mission (40 minutes):** compare an analytical 3D bar scene with a Cesium globe, scrub an invented logistics route, and examine why exaggerated height can change perception. This uses the same CesiumJS family as God's Eye View but is an original, standalone learning scene.


**Runtime:** Python + NumPy + Matplotlib; no model key. Run all cells in order. Interactive viewers additionally load JavaScript from a CDN. All scenario records are fictional.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from gev_lab import *
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
scenario = styled_sectors()
stats = sector_summary()
print('SYNTHETIC TRAINING EXERCISE | snapshot', AS_OF)

## 1. Static 3D fallback: height encodes a rate
Height is a thematic variable, not terrain or building height. A 2D rate map from notebook 06 should accompany it because perspective and occlusion can make comparison harder.

In [ ]:
rates=np.array([s['rate_10000'] for s in stats])
fig=plt.figure(figsize=(10,6));ax=fig.add_subplot(111,projection='3d')
gx,gy=np.meshgrid(np.arange(4),np.arange(3))
ax.bar3d(gx.ravel(),gy.ravel(),np.zeros(12),.8,.8,rates,color=plt.cm.viridis(rates/rates.max()),shade=True)
ax.set(xlabel='Training grid column',ylabel='Training grid row',zlabel='Reported cases / 10,000',title='Fictional sectors · same data, different viewing geometry')
ax.view_init(30,-60);plt.show()

## 2. Interactive Cesium scene
Switch 2D/3D, change the height scale, play/pause the route, and scrub the time strip. The grid globe uses an ellipsoid: **no photorealistic 3D, real terrain, Cesium ion token, or live tracked vehicles**. CDN access and WebGL are required. The scene explicitly labels its invented 5 km-high route.

In [ ]:
show_map(scenario,mode='3d',height=720)
download('riverbend-3d.html',map_page(scenario,'3d'),'text/html')

## 3. A deliberately simple terrain experiment
The water surface below is a horizontal threshold over an invented elevation grid. It does not enforce hydraulic connectivity, flow, drainage, or conservation of water; it is not a flood model.

In [ ]:
x=np.linspace(-20,20,65);y=np.linspace(-15,15,55);X,Y=np.meshgrid(x,y)
Z=110+25*np.sin(X/8)*np.cos(Y/7)+.4*X**2
water_level_m=120
fig=plt.figure(figsize=(11,5));ax=fig.add_subplot(111,projection='3d')
ax.plot_surface(X,Y,Z,cmap='terrain',alpha=.9,linewidth=0)
ax.plot_surface(X,Y,np.full_like(Z,water_level_m),color='#369bd2',alpha=.28)
ax.set(xlabel='Local x (km)',ylabel='Local y (km)',zlabel='Invented elevation (m)',title='Synthetic terrain + horizontal water plane · not hydrodynamics')
plt.show()
print('Grid cells below threshold:',round(float((Z<water_level_m).mean()*100),1),'%')

## Lab challenge
Set thematic column scale to zero: which spatial relationships become easier to see? Rotate the static scene and describe occlusion. Raise the water plane by 10 m; identify a disconnected depression that a simple threshold might incorrectly classify as flooded.

**God's Eye View extension:** the integration adapter in notebook 12 loads the same GeoJSON in a local upstream viewer. Real terrain/photorealistic tiles require separate provider configuration and terms.

**Sources:** [Cesium Viewer](https://cesium.com/learn/cesiumjs/ref-doc/Viewer.html), [GeoJsonDataSource](https://cesium.com/learn/cesiumjs/ref-doc/GeoJsonDataSource.html), [SampledPositionProperty](https://cesium.com/learn/cesiumjs/ref-doc/SampledPositionProperty.html), [upstream license](https://github.com/bilawalsidhu/gods-eye-view/blob/main/LICENSE).